# Cài đặt thư viện và Import

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
!pip install transformers datasets scikit-learn

import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, set_seed
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Đọc dữ liệu và Ánh xạ nhãn

In [11]:
# 1. Đọc dữ liệu

train_df = pd.read_csv('/content/train.csv')
val_df = pd.read_csv('/content/val.csv')
test_df = pd.read_csv('/content/test.csv')

# 2. Map nhãn chuẩn: fake=0, real=1
label_map = {'fake': 0, 'real': 1}
for df in [train_df, val_df, test_df]:
    df['label'] = df['label'].map(label_map)

# 3. Chuyển đổi sang format Dataset của Hugging Face
train_dataset = Dataset.from_pandas(train_df[['tweet', 'label']])
val_dataset = Dataset.from_pandas(val_df[['tweet', 'label']])
test_dataset = Dataset.from_pandas(test_df[['tweet', 'label']])

# Tokenizer(Tiền xử lý văn bản)

In [12]:
# Tải tokenizer của BERT-base
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    # Padding và cắt gọn văn bản
    return tokenizer(examples['tweet'], padding='max_length', truncation=True, max_length=128)

# Áp dụng tokenizer lên toàn bộ tập dữ liệu
train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/6420 [00:00<?, ? examples/s]

Map:   0%|          | 0/2140 [00:00<?, ? examples/s]

Map:   0%|          | 0/2140 [00:00<?, ? examples/s]

# Thiết lập đánh giá và Vòng lặp huấn luyện với 3 Seed

In [13]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_weighted': f1_score(labels, predictions, average='weighted')
    }

seeds = [42, 123, 2024]

# Cấu hình ánh xạ nhãn để tích hợp thẳng vào trọng số của mô hình
id2label = {0: "Fake", 1: "Real"}
label2id = {"Fake": 0, "Real": 1}

for seed in seeds:
    print(f"\n{'='*40}")
    print(f"BẮT ĐẦU HUẤN LUYỆN VỚI SEED: {seed}")
    print(f"{'='*40}")

    set_seed(seed)

    # Đưa id2label và label2id vào kiến trúc model
    model = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=2,
        id2label=id2label,
        label2id=label2id
    )

    output_dir = f'./bert_base_seed_{seed}'

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        metric_for_best_model="f1_weighted",
        load_best_model_at_end=True,
        seed=seed
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        compute_metrics=compute_metrics
    )

    trainer.train()

    val_results = trainer.evaluate()
    print(f"\nKết quả trên tập Validation (Seed {seed}):")
    print(val_results)


BẮT ĐẦU HUẤN LUYỆN VỚI SEED: 42


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,No log,0.107848,0.969626,0.969599
2,0.171201,0.119319,0.970561,0.970544
3,0.053049,0.134294,0.970561,0.970544


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,F1 Weighted
0.055784,0.128299,3,0.971963,0.971939



Kết quả trên tập Validation (Seed 2024):
{'eval_loss': 0.12829886376857758, 'eval_accuracy': 0.9719626168224299, 'eval_f1_weighted': 0.9719386516714464}


# Error Analysis (Phân tích lỗi)

In [14]:
import numpy as np

print("Đang tiến hành phân tích lỗi trên tập Validation...")
# Dự đoán đồng loạt bằng trainer
predictions, labels, _ = trainer.predict(val_tokenized)
preds = np.argmax(predictions, axis=1)

# Lọc ra các vị trí dự đoán sai
error_indices = np.where(preds != labels)[0]
print(f"Tổng số câu dự đoán sai: {len(error_indices)}\n")

# In ra phân tích 10 mẫu đầu tiên
for idx, i in enumerate(error_indices[:10]):
    tweet = val_dataset[int(i)]['tweet']
    true_label = val_dataset[int(i)]['label']
    pred_label = preds[i]

    true_text = "Real (1)" if true_label == 1 else "Fake (0)"
    pred_text = "Real (1)" if pred_label == 1 else "Fake (0)"

    print(f"--- Mẫu sai số {idx + 1} ---")
    print(f"Thực tế: {true_text} | Mô hình đoán: {pred_text}")
    print(f"Nội dung: {tweet}\n")

Đang tiến hành phân tích lỗi trên tập Validation...


Tổng số câu dự đoán sai: 60

--- Mẫu sai số 1 ---
Thực tế: Real (1) | Mô hình đoán: Fake (0)
Nội dung: Tomorrow April 6 we will pass 10000 coronavirus deaths. We passed 5000 on April 2. We passed 1000 on March 26. We passed 100 on March 18.

--- Mẫu sai số 2 ---
Thực tế: Fake (0) | Mô hình đoán: Real (1)
Nội dung: A coronavirus vaccine: Where does it stand? Experts we checked with remain optimistic about a vaccine in the first quarter of 2021. https://t.co/XAwbq9IqrH https://t.co/ONglQMMwFc

--- Mẫu sai số 3 ---
Thực tế: Fake (0) | Mô hình đoán: Real (1)
Nội dung: The Global #dietarysupplements market is anticipated to grow at a #CAGR of around 7.50% during 2020-25. 
Check the full report:- https://t.co/71J6LqCvgr

#food #beverage #marketresearch #marketstrategy #USA #animalhusbandry #covid_19 #kerrygroup #nutrients #Immunity #Energy https://t.co/XPFXA1zzo7

--- Mẫu sai số 4 ---
Thực tế: Fake (0) | Mô hình đoán: Real (1)
Nội dung: WHO reports record daily increase in global coronavirus

# ĐÁNH GIÁ TRÊN TẬP TEST CHO 3 SEED

In [15]:
import pandas as pd
from transformers.trainer_utils import get_last_checkpoint

print("="*50)
print("TIẾN HÀNH ĐÁNH GIÁ TRÊN TẬP TEST (test.csv)")
print("="*50)

test_results = []

for seed in seeds:
    output_dir = f'./bert_base_seed_{seed}'

    checkpoint_dir = get_last_checkpoint(output_dir)
    if checkpoint_dir is None:
        checkpoint_dir = output_dir

    print(f"\nĐang nạp mô hình (Seed {seed}) từ: {checkpoint_dir}")

    model_test = BertForSequenceClassification.from_pretrained(checkpoint_dir)
    model_test.to('cuda' if torch.cuda.is_available() else 'cpu')

    trainer_test = Trainer(
        model=model_test,
        compute_metrics=compute_metrics
    )

    # 1. Đánh giá tổng quát
    res = trainer_test.evaluate(test_tokenized)

    # 2. Suy luận chi tiết để vẽ ma trận nhầm lẫn
    predictions, labels, _ = trainer_test.predict(test_tokenized)
    preds = np.argmax(predictions, axis=1)

    print(f"--> Kết quả Test (Seed {seed}): Accuracy: {res['eval_accuracy']:.4f} | F1: {res['eval_f1_weighted']:.4f}")

    print("\nBáo cáo chi tiết từng lớp (Classification Report):")
    print(classification_report(labels, preds, target_names=["Fake (0)", "Real (1)"], digits=4))

    print("Ma trận nhầm lẫn (Confusion Matrix):")
    print(confusion_matrix(labels, preds))
    print("-" * 50)

    test_results.append({
        'Seed': seed,
        'Accuracy': res['eval_accuracy'],
        'F1 Weighted': res['eval_f1_weighted']
    })

df_test = pd.DataFrame(test_results)
print("\nBẢNG TỔNG HỢP KẾT QUẢ TEST:")
print(df_test)
print(f"\n=> F1 Weighted Trung bình: {df_test['F1 Weighted'].mean():.6f} ± {df_test['F1 Weighted'].std():.6f}")

TIẾN HÀNH ĐÁNH GIÁ TRÊN TẬP TEST (test.csv)

Đang nạp mô hình (Seed 42) từ: ./bert_base_seed_42/checkpoint-1206


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,Accuracy,F1 Weighted
No log,0.109017,0,0.973832,0.973827


--> Kết quả Test (Seed 42): Accuracy: 0.9738 | F1: 0.9738

Báo cáo chi tiết từng lớp (Classification Report):
              precision    recall  f1-score   support

    Fake (0)     0.9763    0.9686    0.9724      1020
    Real (1)     0.9716    0.9786    0.9751      1120

    accuracy                         0.9738      2140
   macro avg     0.9740    0.9736    0.9738      2140
weighted avg     0.9738    0.9738    0.9738      2140

Ma trận nhầm lẫn (Confusion Matrix):
[[ 988   32]
 [  24 1096]]
--------------------------------------------------

Đang nạp mô hình (Seed 123) từ: ./bert_base_seed_123/checkpoint-1206


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,Accuracy,F1 Weighted
No log,0.112311,0,0.974766,0.974752


--> Kết quả Test (Seed 123): Accuracy: 0.9748 | F1: 0.9748

Báo cáo chi tiết từng lớp (Classification Report):
              precision    recall  f1-score   support

    Fake (0)     0.9840    0.9627    0.9732      1020
    Real (1)     0.9667    0.9857    0.9761      1120

    accuracy                         0.9748      2140
   macro avg     0.9753    0.9742    0.9747      2140
weighted avg     0.9749    0.9748    0.9748      2140

Ma trận nhầm lẫn (Confusion Matrix):
[[ 982   38]
 [  16 1104]]
--------------------------------------------------

Đang nạp mô hình (Seed 2024) từ: ./bert_base_seed_2024/checkpoint-1206


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,Accuracy,F1 Weighted
No log,0.123955,0,0.973832,0.973815


--> Kết quả Test (Seed 2024): Accuracy: 0.9738 | F1: 0.9738

Báo cáo chi tiết từng lớp (Classification Report):
              precision    recall  f1-score   support

    Fake (0)     0.9839    0.9608    0.9722      1020
    Real (1)     0.9650    0.9857    0.9753      1120

    accuracy                         0.9738      2140
   macro avg     0.9745    0.9732    0.9737      2140
weighted avg     0.9740    0.9738    0.9738      2140

Ma trận nhầm lẫn (Confusion Matrix):
[[ 980   40]
 [  16 1104]]
--------------------------------------------------

BẢNG TỔNG HỢP KẾT QUẢ TEST:
   Seed  Accuracy  F1 Weighted
0    42  0.973832     0.973827
1   123  0.974766     0.974752
2  2024  0.973832     0.973815

=> F1 Weighted Trung bình: 0.974131 ± 0.000537


# THỬ NGHIỆM VÀ PHÂN TÍCH CÂU THỰC TẾ TRÊN MẠNG

In [16]:

from transformers.trainer_utils import get_last_checkpoint

print("\n" + "="*50)
print("THỬ NGHIỆM MÔ HÌNH VỚI DỮ LIỆU THỰC TẾ TỪ INTERNET")
print("="*50)

# Tự động tìm thư mục checkpoint của seed 2024 (seed tốt nhất)
best_seed_dir = './bert_base_seed_2024'
best_checkpoint = get_last_checkpoint(best_seed_dir)
if best_checkpoint is None:
    best_checkpoint = best_seed_dir

print(f"Đang nạp mô hình từ: {best_checkpoint}")

# Nạp mô hình
best_model = BertForSequenceClassification.from_pretrained(best_checkpoint)
best_model.to('cuda' if torch.cuda.is_available() else 'cpu')

# Bộ dữ liệu 7 câu thực tế ngoài tập Train/Val/Test
internet_samples = [
    {
        "text": "According to the CDC, COVID-19 vaccines are safe and effective. Millions of people have received COVID-19 vaccines under the most intense safety monitoring in U.S. history.",
        "source": "CDC Official Website",
        "true_label": 1 # Real
    },
    {
        "text": "The WHO states that antibiotics do not work against viruses, they only work on bacterial infections. Antibiotics should not be used to treat COVID-19.",
        "source": "WHO Mythbusters",
        "true_label": 1 # Real
    },
    {
        "text": "Holding your breath for 10 seconds or more without coughing or feeling discomfort means you are free from the coronavirus disease.",
        "source": "Internet Myth (Bác bỏ bởi WHO)",
        "true_label": 0 # Fake
    },
    {
        "text": "COVID-19 vaccines contain microchips to track people's movements and control populations via 5G towers.",
        "source": "Conspiracy Theory / Reuters Fact Check",
        "true_label": 0 # Fake
    },
    {
        "text": "Drinking alcohol or spraying chlorine all over your body will kill the new coronavirus.",
        "source": "Internet Rumor",
        "true_label": 0 # Fake
    },
    # 2 CÂU ĐẶC BIỆT MÀ BÁO CÁO CỦA NHÓM ĐÃ CHỈ RA LỖI TỪ DISTILBERT
    {
        "text": "People of all ages can get infected by the new coronavirus.",
        "source": "WHO Viet Nam",
        "true_label": 1 # Real
    },
    {
        "text": "No, drinking alcohol does not protect you from COVID-19 infection.",
        "source": "WHO Viet Nam",
        "true_label": 1 # Real
    }
]

def test_wild_samples(samples, model, tokenizer):
    model.eval()
    wrong_preds = []

    for i, sample in enumerate(samples):
        text = sample["text"]
        true_lbl = sample["true_label"]
        src = sample["source"]

        inputs = tokenizer(text, padding='max_length', truncation=True, max_length=128, return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            pred_lbl = torch.argmax(outputs.logits, axis=-1).item()

        true_name = "Real (1)" if true_lbl == 1 else "Fake (0)"
        pred_name = "Real (1)" if pred_lbl == 1 else "Fake (0)"
        status = "✅ ĐÚNG" if true_lbl == pred_lbl else "❌ SAI"

        print(f"Mẫu {i+1} [{status}]: {text}")
        print(f"Nguồn: {src} | Nhãn thực tế: {true_name} | Dự đoán: {pred_name}\n")

        if true_lbl != pred_lbl:
            wrong_preds.append((sample, pred_name))

    return wrong_preds

wrong_predictions = test_wild_samples(internet_samples, best_model, tokenizer)

# -- TỔNG KẾT PHÂN TÍCH LỖI --
print("-" * 50)
print("PHÂN TÍCH LỖI TRÊN CÁC CÂU NGOÀI DỮ LIỆU:")
if len(wrong_predictions) == 0:
    print("Mô hình dự đoán đúng 100% các câu thực tế ngoài dữ liệu.")
else:
    for sample, pred in wrong_predictions:
        print(f"-> Text: {sample['text']}")
        print(f"   Lỗi: Mô hình đoán {pred} trong khi thực tế là {'Real' if sample['true_label']==1 else 'Fake'}.")
        print("   Phân tích: Câu này lấy từ nguồn thực tế nhưng không chứa các dấu hiệu quen thuộc trong tập Train (như link URL t.co, hashtag). Việc thiếu metadata hoặc sử dụng cấu trúc phủ định ('No', 'does not') khiến mô hình bị nhầm lẫn giữa phong cách văn bản và độ xác thực thông tin.\n")


THỬ NGHIỆM MÔ HÌNH VỚI DỮ LIỆU THỰC TẾ TỪ INTERNET
Đang nạp mô hình từ: ./bert_base_seed_2024/checkpoint-1206


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Mẫu 1 [❌ SAI]: According to the CDC, COVID-19 vaccines are safe and effective. Millions of people have received COVID-19 vaccines under the most intense safety monitoring in U.S. history.
Nguồn: CDC Official Website | Nhãn thực tế: Real (1) | Dự đoán: Fake (0)

Mẫu 2 [❌ SAI]: The WHO states that antibiotics do not work against viruses, they only work on bacterial infections. Antibiotics should not be used to treat COVID-19.
Nguồn: WHO Mythbusters | Nhãn thực tế: Real (1) | Dự đoán: Fake (0)

Mẫu 3 [✅ ĐÚNG]: Holding your breath for 10 seconds or more without coughing or feeling discomfort means you are free from the coronavirus disease.
Nguồn: Internet Myth (Bác bỏ bởi WHO) | Nhãn thực tế: Fake (0) | Dự đoán: Fake (0)

Mẫu 4 [✅ ĐÚNG]: COVID-19 vaccines contain microchips to track people's movements and control populations via 5G towers.
Nguồn: Conspiracy Theory / Reuters Fact Check | Nhãn thực tế: Fake (0) | Dự đoán: Fake (0)

Mẫu 5 [✅ ĐÚNG]: Drinking alcohol or spraying chlorine all ov

In [17]:
# ---DEMO INFERENCE---
def predict_fake_news(text):
    print(f"Đang phân tích câu: '{text}'")

    inputs = tokenizer(text, padding='max_length', truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(best_model.device) for k, v in inputs.items()}

    best_model.eval()
    with torch.no_grad():
        outputs = best_model(**inputs)

    prediction = torch.argmax(outputs.logits, axis=-1).item()

    # Tự động lấy nhãn từ metadata đã cấu hình ở bước Train
    label_name = f"{best_model.config.id2label[prediction]} ({'Tin thật' if prediction == 1 else 'Tin giả'})"
    print(f"=> Kết quả dự đoán: {label_name}\n")

cau_thu_nghiem = "States reported 1121 deaths a small rise from last Tuesday. Southern states reported 640 of those deaths. https://t.co/YASGRTT4ux,"
predict_fake_news(cau_thu_nghiem)

Đang phân tích câu: 'States reported 1121 deaths a small rise from last Tuesday. Southern states reported 640 of those deaths. https://t.co/YASGRTT4ux,'
=> Kết quả dự đoán: Real (Tin thật)

